# Azure Machine Learning Training
## Lab: Train, Register, and Deploy a Model

### Objective
By the end of this lab, you will:
- Connect to Azure ML workspace
- Train a scikit-learn model on Azure compute
- Register the model in the workspace
- Deploy the model as a REST endpoint
- Test the endpoint with sample data

---

## Lab 1: Install Dependencies

In [ ]:
!pip install azure-ai-ml azure-identity scikit-learn mlflow

## Lab 2: Connect to Azure ML Workspace

In [ ]:
from azure.ai.ml import MLClient, command
from azure.identity import DefaultAzureCredential

# TODO: Replace these with your values
SUBSCRIPTION_ID = "<YOUR-SUBSCRIPTION-ID>"  # Find in Azure Portal → Subscriptions
RESOURCE_GROUP = "<YOUR-RESOURCE-GROUP>"   # e.g., "Kedir.Yassin-rg"
WORKSPACE_NAME = "<YOUR-WORKSPACE>"       # e.g., "mlw-training"

# Connect to workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME
)
print("✅ Connected to workspace!")
print(f"Workspace: {WORKSPACE_NAME}")
print(f"Resource Group: {RESOURCE_GROUP}")

## Lab 3: Write the Training Script

In [ ]:
%%writefile train_iris.py
import joblib
import mlflow
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load data
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Log with MLflow
mlflow.log_param("n_estimators", 100)
mlflow.log_param("random_state", 42)
mlflow.log_metric("accuracy", accuracy)
mlflow.log_metric("test_size", 0.2)

# Save model
joblib.dump(model, "model.pkl")
print(f"✅ Model trained! Accuracy: {accuracy:.4f}")
print("✅ Model saved as model.pkl")

## Lab 4: Submit Training Job to Azure

In [ ]:
# Configure the job
job = command(
    code=".",
    command="python train_iris.py",
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute="cpu-cluster",  # Must match your compute cluster name
    display_name="iris-training"
)

# Submit the job
returned_job = ml_client.jobs.create_or_update(job)
print(f"✅ Job submitted: {returned_job.name}")
print(f"📊 View job at: {returned_job.studio_url}")

## Lab 5: Monitor Job Status

In [ ]:
import time

job_status = ml_client.jobs.get(returned_job.name)
print(f"Job: {job_status.name}")
print(f"Status: {job_status.status}")

# Wait for completion (if needed)
print("⏳ Waiting for job to complete...")
while job_status.status not in ["Completed", "Failed", "Canceled"]:
    time.sleep(10)
    job_status = ml_client.jobs.get(returned_job.name)
    print(f"Status: {job_status.status}")

print(f"✅ Job completed with status: {job_status.status}")

# Check metrics
if job_status.status == "Completed":
    print(f"📊 Metrics: {job_status.metrics}")

## Lab 6: Register the Model

In [ ]:
from azure.ai.ml.entities import Model

model = Model(
    name="iris-classifier",
    path=returned_job.outputs.model,  # The model output from the job
    type="mlflow_model",
    description="Iris classifier using Random Forest"
)

registered_model = ml_client.models.create_or_update(model)
print(f"✅ Model registered: {registered_model.name}")
print(f"📌 Version: {registered_model.version}")

## Lab 7: Deploy Model as Endpoint

**⚠️ Note**: This step takes 5-15 minutes. Please be patient.

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment

# Create a unique endpoint name
endpoint_name = "iris-endpoint-" + str(hash(ml_client.workspace_name))[:6]
print(f"Endpoint name: {endpoint_name}")

# Create endpoint
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Iris classifier endpoint",
    auth_mode="key"
)

ml_client.online_endpoints.begin_create_or_update(endpoint).wait()
print("✅ Endpoint created!")

In [ ]:
# Create deployment
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=registered_model.name,
    instance_type="Standard_DS2_v2",
    instance_count=1,
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest"
)

print("⏳ Deploying... This may take 5-15 minutes.")
ml_client.online_deployments.begin_create_or_update(deployment).wait()
print("✅ Deployment created!")

In [ ]:
# Set the deployment as default (traffic = 100%)
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).wait()
print("✅ Endpoint ready for inference!")

## Lab 8: Test the Endpoint

In [ ]:
import json

# Create sample input
test_input = {
    "input_data": [
        [5.1, 3.5, 1.4, 0.2],   # Setosa
        [6.5, 3.0, 5.8, 2.2],   # Virginica
        [5.9, 3.2, 4.8, 1.8]    # Versicolor
    ]
}

# Save to file
with open("sample.json", "w") as f:
    json.dump(test_input, f)
print("✅ Sample input saved to sample.json")

# Invoke endpoint
response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    request_file="sample.json"
)

print("✅ Prediction:")
print(f"{response}")

# Interpret results
iris_classes = ['Setosa', 'Versicolor', 'Virginica']
predictions = eval(response)  # Convert string to list
print("\n📊 Interpreted Results:")
for i, pred in enumerate(predictions):
    print(f"Sample {i+1}: {iris_classes[pred]}")

## Lab 9: Model Management

### View all registered models

In [ ]:
models = ml_client.models.list()
print("📚 Registered Models:")
for m in models:
    print(f"  - {m.name} (Version: {m.version})")

### Get endpoint details

In [ ]:
endpoint_info = ml_client.online_endpoints.get(name=endpoint_name)
print(f"🔗 Endpoint: {endpoint_name}")
print(f"   - State: {endpoint_info.provisioning_state}")
print(f"   - Auth Mode: {endpoint_info.auth_mode}")
print(f"   - URI: {endpoint_info.scoring_uri}")

## Lab 10: Cleanup (Optional)

**⚠️ Important**: Delete the endpoint to avoid unnecessary charges.

In [ ]:
# Delete endpoint (saves costs!)
ml_client.online_endpoints.begin_delete(name=endpoint_name).wait()
print("✅ Endpoint deleted!")
print("💡 You can always redeploy if needed.")

## Summary

Congratulations! You have completed the Azure ML training lab.

### What You Learned:
| Topic | Description |
|-------|-------------|
| **Connecting to Azure ML** | Using MLClient and DefaultAzureCredential |
| **Training** | Submitting jobs to cloud compute clusters |
| **Model Registry** | Registering and versioning models |
| **Deployment** | Creating online endpoints for real-time inference |
| **Testing** | Invoking endpoints with JSON input |

### Next Steps:
- Explore AutoML in Azure ML Studio
- Try deploying to a different endpoint type (Batch, AKS)
- Integrate with other Azure services (Functions, Logic Apps)

**🎉 Great work!**